# 04 — Training the Scientific Abstract GPT

This notebook:

- Loads the BPE token streams
- Reconstructs the GPT model
- Trains the model using next-token prediction
- Uses mixed precision on GPU
- Uses warmup and cosine learning-rate decay
- Saves the best and final checkpoints
- Saves training history for evaluation and text generation

In [1]:
from google.colab import drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


In [2]:
import os
import sys
import json
import math
import time
import random
import inspect

import numpy as np
import torch

In [5]:
PROJECT_PATH = (
    "/content/drive/MyDrive/"
    "Scientific-Abstract-GPT"
)

SRC_FOLDER = os.path.join(
    PROJECT_PATH,
    "src"
)

DATA_FOLDER = os.path.join(
    PROJECT_PATH,
    "data"
)

TOKEN_STREAM_FOLDER = os.path.join(
    DATA_FOLDER,
    "tokenized_bpe_streams"
)

TOKENIZER_FOLDER = os.path.join(
    DATA_FOLDER,
    "bpe_tokenizer"
)

TOKENIZER_PATH = os.path.join(
    TOKENIZER_FOLDER,
    "tokenizer.json"
)

MODEL_FOLDER = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_FOLDER = os.path.join(
    PROJECT_PATH,
    "outputs"
)

MODEL_CONFIG_PATH = os.path.join(
    MODEL_FOLDER,
    "gpt_model_config.json"
)

TRAIN_STREAM_PATH = os.path.join(
    TOKEN_STREAM_FOLDER,
    "train_tokens.bin"
)

VALIDATION_STREAM_PATH = os.path.join(
    TOKEN_STREAM_FOLDER,
    "validation_tokens.bin"
)

STREAM_METADATA_PATH = os.path.join(
    TOKEN_STREAM_FOLDER,
    "stream_metadata.json"
)

BEST_CHECKPOINT_PATH = os.path.join(
    MODEL_FOLDER,
    "best_model.pt"
)

FINAL_CHECKPOINT_PATH = os.path.join(
    MODEL_FOLDER,
    "final_model.pt"
)

TRAINING_HISTORY_PATH = os.path.join(
    OUTPUT_FOLDER,
    "training_history.json"
)

os.makedirs(
    MODEL_FOLDER,
    exist_ok=True
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

if PROJECT_PATH not in sys.path:

    sys.path.insert(
        0,
        PROJECT_PATH
    )

print(
    "Project path:",
    PROJECT_PATH
)

Project path: /content/drive/MyDrive/Scientific-Abstract-GPT


In [6]:
required_files = {
    "GPT model configuration": MODEL_CONFIG_PATH,
    "training token stream": TRAIN_STREAM_PATH,
    "validation token stream": VALIDATION_STREAM_PATH,
    "stream metadata": STREAM_METADATA_PATH,
    "tokenizer": TOKENIZER_PATH
}

missing_files = []

for file_name, file_path in required_files.items():

    exists = os.path.exists(
        file_path
    )

    print(
        f"{file_name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    if not exists:

        missing_files.append(
            file_path
        )

if missing_files:

    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(missing_files)
    )

print(
    "\nAll required files are available."
)

GPT model configuration: FOUND
training token stream: FOUND
validation token stream: FOUND
stream metadata: FOUND
tokenizer: FOUND

All required files are available.


Import Common GPT Components

In [7]:
from src.gpt_components import (
    GPTConfig,
    GPTLanguageModel,
    set_seed,
    count_parameters
)

print(
    "GPT components imported successfully."
)

GPT components imported successfully.


In [8]:
SEED = 42

set_seed(
    SEED
)

device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Selected device:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Selected device: cuda:0
GPU: Tesla T4


In [9]:
with open(
    MODEL_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as file:

    model_config_dictionary = json.load(
        file
    )

model_config = GPTConfig.from_dict(
    model_config_dictionary
)

print(
    model_config
)

GPTConfig(vocab_size=8000, block_size=256, n_embd=256, n_head=8, n_layer=6, dropout=0.1, bias=True)


Load Token Streams

In [10]:
with open(
    STREAM_METADATA_PATH,
    "r",
    encoding="utf-8"
) as file:

    stream_metadata = json.load(
        file
    )

stream_dtype_name = stream_metadata.get(
    "dtype",
    "uint16"
)

stream_dtype = np.dtype(
    stream_dtype_name
)

train_tokens = np.memmap(
    TRAIN_STREAM_PATH,
    dtype=stream_dtype,
    mode="r"
)

validation_tokens = np.memmap(
    VALIDATION_STREAM_PATH,
    dtype=stream_dtype,
    mode="r"
)

print(
    "Training tokens:",
    f"{len(train_tokens):,}"
)

print(
    "Validation tokens:",
    f"{len(validation_tokens):,}"
)

print(
    "Token-stream dtype:",
    stream_dtype
)

Training tokens: 45,356,215
Validation tokens: 2,515,850
Token-stream dtype: uint16


In [11]:
minimum_required_tokens = (
    model_config.block_size + 2
)

if len(train_tokens) < minimum_required_tokens:

    raise ValueError(
        "Training token stream is too short."
    )

if len(validation_tokens) < minimum_required_tokens:

    raise ValueError(
        "Validation token stream is too short."
    )

maximum_train_token = int(
    np.max(train_tokens)
)

maximum_validation_token = int(
    np.max(validation_tokens)
)

if maximum_train_token >= model_config.vocab_size:

    raise ValueError(
        "Training stream contains a token ID "
        "outside the tokenizer vocabulary."
    )

if maximum_validation_token >= model_config.vocab_size:

    raise ValueError(
        "Validation stream contains a token ID "
        "outside the tokenizer vocabulary."
    )

print(
    "Token-stream validation passed."
)

Token-stream validation passed.


In [12]:
BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2

MAX_STEPS = 3000

EVAL_INTERVAL = 200
EVAL_BATCHES = 50

LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 3e-5
WARMUP_STEPS = 200

WEIGHT_DECAY = 0.1
BETA_1 = 0.9
BETA_2 = 0.95

GRADIENT_CLIP = 1.0

USE_AMP = (
    device.type == "cuda"
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Effective batch size:",
    BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

print(
    "Maximum training steps:",
    MAX_STEPS
)

print(
    "Mixed precision enabled:",
    USE_AMP
)

Batch size: 32
Effective batch size: 64
Maximum training steps: 3000
Mixed precision enabled: True


Batch Creation Function

In [13]:
def get_batch(
    split
):

    if split == "train":

        data = train_tokens

    elif split == "validation":

        data = validation_tokens

    else:

        raise ValueError(
            "split must be 'train' "
            "or 'validation'."
        )

    maximum_start_index = (
        len(data)
        - model_config.block_size
        - 1
    )

    start_indices = torch.randint(
        low=0,
        high=maximum_start_index,
        size=(BATCH_SIZE,)
    )

    input_sequences = []

    target_sequences = []

    for start_index in start_indices.tolist():

        input_array = np.asarray(
            data[
                start_index:
                start_index
                + model_config.block_size
            ],
            dtype=np.int64
        )

        target_array = np.asarray(
            data[
                start_index + 1:
                start_index
                + model_config.block_size
                + 1
            ],
            dtype=np.int64
        )

        input_sequences.append(
            torch.from_numpy(
                input_array.copy()
            )
        )

        target_sequences.append(
            torch.from_numpy(
                target_array.copy()
            )
        )

    input_batch = torch.stack(
        input_sequences
    )

    target_batch = torch.stack(
        target_sequences
    )

    input_batch = input_batch.to(
        device,
        non_blocking=True
    )

    target_batch = target_batch.to(
        device,
        non_blocking=True
    )

    return (
        input_batch,
        target_batch
    )

Test Batch Creation

In [14]:
test_inputs, test_targets = get_batch(
    "train"
)

print(
    "Input batch shape:",
    test_inputs.shape
)

print(
    "Target batch shape:",
    test_targets.shape
)

print(
    "Input device:",
    test_inputs.device
)

print(
    "Target device:",
    test_targets.device
)

print(
    "Maximum input token:",
    test_inputs.max().item()
)

Input batch shape: torch.Size([32, 256])
Target batch shape: torch.Size([32, 256])
Input device: cuda:0
Target device: cuda:0
Maximum input token: 7982


Initialize the Model

In [15]:
if "model" in locals() and model is not None:

    del model

if torch.cuda.is_available():

    torch.cuda.empty_cache()

model = GPTLanguageModel(
    model_config
).to(device)

model_device = next(
    model.parameters()
).device

model_on_selected_device = (
    model_device.type == device.type
)

if device.type == "cuda":

    selected_device_index = (
        0
        if device.index is None
        else device.index
    )

    model_device_index = (
        0
        if model_device.index is None
        else model_device.index
    )

    model_on_selected_device = (
        model_on_selected_device
        and selected_device_index
        == model_device_index
    )

if not model_on_selected_device:

    raise RuntimeError(
        "Model is not on the selected device."
    )

print(
    "Model device:",
    model_device
)

print(
    "Trainable parameters:",
    f"{count_parameters(model):,}"
)

Model device: cuda:0
Trainable parameters: 6,852,608


Create Optimizer and Mixed-Precision Scaler

In [16]:
optimizer_arguments = {
    "params": model.parameters(),
    "lr": LEARNING_RATE,
    "betas": (
        BETA_1,
        BETA_2
    ),
    "weight_decay": WEIGHT_DECAY
}

adamw_parameters = inspect.signature(
    torch.optim.AdamW
).parameters

if (
    "fused" in adamw_parameters
    and device.type == "cuda"
):

    optimizer_arguments[
        "fused"
    ] = True

optimizer = torch.optim.AdamW(
    **optimizer_arguments
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(
    "Optimizer created:",
    optimizer.__class__.__name__
)

print(
    "AMP scaler enabled:",
    scaler.is_enabled()
)

Optimizer created: AdamW
AMP scaler enabled: True


Learning-Rate Schedule

In [17]:
def get_learning_rate(
    step
):

    if step < WARMUP_STEPS:

        return (
            LEARNING_RATE
            * (step + 1)
            / WARMUP_STEPS
        )

    if step >= MAX_STEPS:

        return MIN_LEARNING_RATE

    decay_ratio = (
        step - WARMUP_STEPS
    ) / (
        MAX_STEPS - WARMUP_STEPS
    )

    cosine_coefficient = (
        0.5
        * (
            1.0
            + math.cos(
                math.pi
                * decay_ratio
            )
        )
    )

    return (
        MIN_LEARNING_RATE
        + cosine_coefficient
        * (
            LEARNING_RATE
            - MIN_LEARNING_RATE
        )
    )


print(
    "Initial learning rate:",
    get_learning_rate(0)
)

print(
    "Peak learning rate:",
    get_learning_rate(
        WARMUP_STEPS
    )
)

print(
    "Final learning rate:",
    get_learning_rate(
        MAX_STEPS
    )
)

Initial learning rate: 1.4999999999999998e-06
Peak learning rate: 0.0003
Final learning rate: 3e-05


Loss Estimation Function

In [18]:
@torch.no_grad()
def estimate_loss():

    model.eval()

    results = {}

    for split in [
        "train",
        "validation"
    ]:

        losses = torch.zeros(
            EVAL_BATCHES
        )

        for batch_index in range(
            EVAL_BATCHES
        ):

            inputs, targets = get_batch(
                split
            )

            with torch.amp.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                _, loss = model(
                    inputs,
                    targets
                )

            losses[
                batch_index
            ] = loss.detach().cpu()

        results[
            split
        ] = losses.mean().item()

    model.train()

    return results

In [19]:
def save_checkpoint(
    checkpoint_path,
    step,
    train_loss,
    validation_loss,
    best_validation_loss
):

    checkpoint = {
        "step": step,
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "scaler_state_dict": (
            scaler.state_dict()
        ),
        "model_config": (
            model_config.to_dict()
        ),
        "train_loss": (
            float(train_loss)
        ),
        "validation_loss": (
            float(validation_loss)
        ),
        "best_validation_loss": (
            float(best_validation_loss)
        ),
        "tokenizer_path": (
            TOKENIZER_PATH
        ),
        "seed": SEED
    }

    torch.save(
        checkpoint,
        checkpoint_path
    )

In [20]:
initial_losses = estimate_loss()

print(
    "Initial training loss:",
    f"{initial_losses['train']:.4f}"
)

print(
    "Initial validation loss:",
    f"{initial_losses['validation']:.4f}"
)

print(
    "Initial validation perplexity:",
    f"{math.exp(initial_losses['validation']):.4f}"
)

Initial training loss: 9.0360
Initial validation loss: 9.0363
Initial validation perplexity: 8402.4991


Train the Model

In [21]:
training_history = []

best_validation_loss = float(
    "inf"
)

training_start_time = time.time()

model.train()

for step in range(
    MAX_STEPS + 1
):

    if (
        step % EVAL_INTERVAL == 0
        or step == MAX_STEPS
    ):

        losses = estimate_loss()

        validation_perplexity = math.exp(
            min(
                losses[
                    "validation"
                ],
                20
            )
        )

        elapsed_minutes = (
            time.time()
            - training_start_time
        ) / 60

        history_record = {
            "step": step,
            "train_loss": (
                losses["train"]
            ),
            "validation_loss": (
                losses["validation"]
            ),
            "validation_perplexity": (
                validation_perplexity
            ),
            "learning_rate": (
                optimizer.param_groups[
                    0
                ]["lr"]
            ),
            "elapsed_minutes": (
                elapsed_minutes
            )
        }

        training_history.append(
            history_record
        )

        print(
            f"Step {step:4d} | "
            f"Train Loss {losses['train']:.4f} | "
            f"Val Loss {losses['validation']:.4f} | "
            f"Val PPL {validation_perplexity:.2f} | "
            f"Time {elapsed_minutes:.1f} min"
        )

        if (
            losses["validation"]
            < best_validation_loss
        ):

            best_validation_loss = (
                losses[
                    "validation"
                ]
            )

            save_checkpoint(
                checkpoint_path=(
                    BEST_CHECKPOINT_PATH
                ),
                step=step,
                train_loss=(
                    losses["train"]
                ),
                validation_loss=(
                    losses[
                        "validation"
                    ]
                ),
                best_validation_loss=(
                    best_validation_loss
                )
            )

            print(
                "Best checkpoint saved."
            )

    if step == MAX_STEPS:

        break

    learning_rate = get_learning_rate(
        step
    )

    for parameter_group in (
        optimizer.param_groups
    ):

        parameter_group[
            "lr"
        ] = learning_rate

    optimizer.zero_grad(
        set_to_none=True
    )

    accumulated_loss = 0.0

    for _ in range(
        GRADIENT_ACCUMULATION_STEPS
    ):

        inputs, targets = get_batch(
            "train"
        )

        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            _, loss = model(
                inputs,
                targets
            )

            scaled_loss = (
                loss
                / GRADIENT_ACCUMULATION_STEPS
            )

        accumulated_loss += (
            loss.detach().item()
        )

        scaler.scale(
            scaled_loss
        ).backward()

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRADIENT_CLIP
    )

    scaler.step(
        optimizer
    )

    scaler.update()

Step    0 | Train Loss 9.0357 | Val Loss 9.0359 | Val PPL 8399.33 | Time 0.1 min
Best checkpoint saved.
Step  200 | Train Loss 5.8503 | Val Loss 5.8590 | Val PPL 350.37 | Time 1.0 min
Best checkpoint saved.
Step  400 | Train Loss 5.1704 | Val Loss 5.1766 | Val PPL 177.08 | Time 2.0 min
Best checkpoint saved.
Step  600 | Train Loss 4.8093 | Val Loss 4.8243 | Val PPL 124.49 | Time 3.0 min
Best checkpoint saved.
Step  800 | Train Loss 4.5643 | Val Loss 4.5679 | Val PPL 96.34 | Time 3.9 min
Best checkpoint saved.
Step 1000 | Train Loss 4.3942 | Val Loss 4.4055 | Val PPL 81.90 | Time 4.9 min
Best checkpoint saved.
Step 1200 | Train Loss 4.2591 | Val Loss 4.2810 | Val PPL 72.31 | Time 5.9 min
Best checkpoint saved.
Step 1400 | Train Loss 4.1672 | Val Loss 4.1844 | Val PPL 65.65 | Time 6.9 min
Best checkpoint saved.
Step 1600 | Train Loss 4.0911 | Val Loss 4.0926 | Val PPL 59.90 | Time 7.8 min
Best checkpoint saved.
Step 1800 | Train Loss 4.0391 | Val Loss 4.0622 | Val PPL 58.10 | Time 8.8 mi

Save Final Checkpoint

In [22]:
final_losses = estimate_loss()

save_checkpoint(
    checkpoint_path=(
        FINAL_CHECKPOINT_PATH
    ),
    step=MAX_STEPS,
    train_loss=(
        final_losses["train"]
    ),
    validation_loss=(
        final_losses[
            "validation"
        ]
    ),
    best_validation_loss=(
        best_validation_loss
    )
)

print(
    "Final checkpoint saved:"
)

print(
    FINAL_CHECKPOINT_PATH
)

Final checkpoint saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/models/final_model.pt


Save Training History

In [23]:
training_duration_minutes = (
    time.time()
    - training_start_time
) / 60

training_summary = {
    "model_config": (
        model_config.to_dict()
    ),
    "training_config": {
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": (
            GRADIENT_ACCUMULATION_STEPS
        ),
        "effective_batch_size": (
            BATCH_SIZE
            * GRADIENT_ACCUMULATION_STEPS
        ),
        "max_steps": MAX_STEPS,
        "learning_rate": LEARNING_RATE,
        "minimum_learning_rate": (
            MIN_LEARNING_RATE
        ),
        "warmup_steps": WARMUP_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "gradient_clip": GRADIENT_CLIP,
        "mixed_precision": USE_AMP
    },
    "final_train_loss": (
        final_losses["train"]
    ),
    "final_validation_loss": (
        final_losses["validation"]
    ),
    "final_validation_perplexity": (
        math.exp(
            min(
                final_losses[
                    "validation"
                ],
                20
            )
        )
    ),
    "best_validation_loss": (
        best_validation_loss
    ),
    "training_duration_minutes": (
        training_duration_minutes
    ),
    "best_checkpoint_path": (
        BEST_CHECKPOINT_PATH
    ),
    "final_checkpoint_path": (
        FINAL_CHECKPOINT_PATH
    ),
    "history": training_history
}

with open(
    TRAINING_HISTORY_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        training_summary,
        file,
        indent=2
    )

print(
    "Training history saved:"
)

print(
    TRAINING_HISTORY_PATH
)

Training history saved:
/content/drive/MyDrive/Scientific-Abstract-GPT/outputs/training_history.json


Verify the Best Checkpoint

In [24]:
best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

verification_model = GPTLanguageModel(
    GPTConfig.from_dict(
        best_checkpoint[
            "model_config"
        ]
    )
).to(device)

verification_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

verification_model.eval()

verification_inputs, verification_targets = (
    get_batch(
        "validation"
    )
)

with torch.no_grad():

    with torch.amp.autocast(
        device_type=device.type,
        dtype=torch.float16,
        enabled=USE_AMP
    ):

        verification_logits, verification_loss = (
            verification_model(
                verification_inputs,
                verification_targets
            )
        )

print(
    "Checkpoint step:",
    best_checkpoint["step"]
)

print(
    "Checkpoint validation loss:",
    f"{best_checkpoint['validation_loss']:.4f}"
)

print(
    "Reloaded model loss:",
    f"{verification_loss.item():.4f}"
)

print(
    "Output shape:",
    verification_logits.shape
)

Checkpoint step: 3000
Checkpoint validation loss: 3.8935
Reloaded model loss: 3.9525
Output shape: torch.Size([32, 256, 8000])


Final Training Summary

In [26]:
required_output_files = {
    "best checkpoint": (
        BEST_CHECKPOINT_PATH
    ),
    "final checkpoint": (
        FINAL_CHECKPOINT_PATH
    ),
    "training history": (
        TRAINING_HISTORY_PATH
    )
}

all_outputs_available = True

for output_name, output_path in (
    required_output_files.items()
):

    exists = os.path.exists(
        output_path
    )

    print(
        f"{output_name}: "
        f"{'FOUND' if exists else 'MISSING'}"
    )

    all_outputs_available = (
        all_outputs_available
        and exists
    )

if not all_outputs_available:

    raise RuntimeError(
        "One or more training outputs "
        "were not created."
    )

print(
    "\n" + "=" * 70
)

print(
    "TRAINING COMPLETED SUCCESSFULLY"
)

print(
    "=" * 70
)

print(
    "Final training loss:",
    f"{final_losses['train']:.4f}"
)

print(
    "Final validation loss:",
    f"{final_losses['validation']:.4f}"
)

print(
    "Final validation perplexity:",
    f"{math.exp(min(final_losses['validation'], 20)):.4f}"
)

print(
    "Best validation loss:",
    f"{best_validation_loss:.4f}"
)

print(
    "Training duration:",
    f"{training_duration_minutes:.2f} minutes"
)

print(
    BEST_CHECKPOINT_PATH
)

best checkpoint: FOUND
final checkpoint: FOUND
training history: FOUND

TRAINING COMPLETED SUCCESSFULLY
Final training loss: 3.8820
Final validation loss: 3.8883
Final validation perplexity: 48.8267
Best validation loss: 3.8935
Training duration: 16.42 minutes
/content/drive/MyDrive/Scientific-Abstract-GPT/models/best_model.pt
